# Capítulo 16 — Duplicados en un DataFrame

## Breve repaso

En el trabajo anterior sobre valores faltantes aprendimos que detectar un problema no alcanza para decidir automáticamente cómo resolverlo.

Vimos que los valores faltantes pueden tratarse de distintas maneras: eliminar filas o columnas, imputar valores con algún criterio o reconstruir información a partir de otras columnas. También vimos que cada estrategia tiene riesgos y que toda transformación debe verificarse después de aplicarse.

En este capítulo vamos a trabajar con otro problema frecuente en la limpieza de datos: los duplicados.

Un duplicado aparece cuando una observación parece estar repetida dentro del dataset. A veces se trata de una fila exactamente igual a otra. Otras veces, el duplicado no es completo, pero se repite una columna importante, como un identificador de transacción.

Para estudiar este tema vamos a usar un dataset sintético pequeño. Esto nos permitirá observar con claridad distintos tipos de duplicados y discutir qué decisiones podrían tomarse en cada caso.

El objetivo no será eliminar filas automáticamente, sino aprender a detectar duplicados, interpretarlos y verificar qué ocurre después de tratarlos.

Al finalizar este capítulo deberías poder:

- Comprender qué significa que una fila esté duplicada.
- Diferenciar duplicados completos de duplicados según columnas específicas.
- Usar `duplicated()` para detectar duplicados.
- Contar duplicados dentro de un `DataFrame`.
- Observar las filas duplicadas antes de eliminarlas.
- Usar `drop_duplicates()` para eliminar duplicados.
- Usar `subset` para revisar duplicados según columnas específicas.
- Decidir cuándo conviene eliminar duplicados y cuándo conviene revisarlos con más cuidado.
- Verificar el resultado después de eliminar duplicados.

## Dataset de trabajo

Para estudiar duplicados vamos a usar un dataset pequeño de ventas.

Cada fila representa una transacción. El dataset incluye un identificador de transacción, el producto vendido, la cantidad, el precio unitario, la sucursal y la fecha.

A diferencia de los datasets anteriores, este fue construido especialmente para mostrar distintos casos de duplicados. Algunos registros están repetidos exactamente, otros comparten el mismo identificador de transacción y otros son parecidos pero podrían representar ventas reales distintas.

In [61]:
import pandas as pd

datos = {
    "id_transaccion": [
        "T001", "T002", "T003", "T004", "T004",
        "T005", "T006", "T007", "T008", "T009",
        "T010", "T010", "T011", "T012", "T013"
    ],
    "producto": [
        "Cafe", "Medialuna", "Cafe", "Te", "Te",
        "Tostado", "Cafe", "Jugo", "Cafe", "Medialuna",
        "Tostado", "Tostado", "Cafe", "Cafe", "Te"
    ],
    "cantidad": [
        2, 3, 1, 2, 2,
        1, 2, 1, 2, 3,
        1, 1, 2, 2, 2
    ],
    "precio_unitario": [
        1200, 800, 1200, 900, 900,
        2500, 1200, 1500, 1200, 800,
        2500, 2500, 1200, 1200, 900
    ],
    "sucursal": [
        "Centro", "Centro", "Norte", "Sur", "Sur",
        "Centro", "Centro", "Norte", "Centro", "Centro",
        "Sur", "Sur", "Centro", "Centro", "Sur"
    ],
    "fecha": [
        "2024-04-01", "2024-04-01", "2024-04-01", "2024-04-02", "2024-04-02",
        "2024-04-02", "2024-04-03", "2024-04-03", "2024-04-03", "2024-04-04",
        "2024-04-04", "2024-04-04", "2024-04-05", "2024-04-05", "2024-04-05"
    ]
}

df = pd.DataFrame(datos)

df

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
0,T001,Cafe,2,1200,Centro,2024-04-01
1,T002,Medialuna,3,800,Centro,2024-04-01
2,T003,Cafe,1,1200,Norte,2024-04-01
3,T004,Te,2,900,Sur,2024-04-02
4,T004,Te,2,900,Sur,2024-04-02
5,T005,Tostado,1,2500,Centro,2024-04-02
6,T006,Cafe,2,1200,Centro,2024-04-03
7,T007,Jugo,1,1500,Norte,2024-04-03
8,T008,Cafe,2,1200,Centro,2024-04-03
9,T009,Medialuna,3,800,Centro,2024-04-04


## Qué significa que una fila esté duplicada

Una fila duplicada es una fila que aparece repetida dentro de un `DataFrame`.

En el caso más simple, dos filas son duplicadas cuando todos sus valores son exactamente iguales. Es decir, tienen el mismo identificador, el mismo producto, la misma cantidad, el mismo precio unitario, la misma sucursal y la misma fecha.

Pero en datasets reales la situación puede ser más compleja. A veces dos filas no son idénticas en todas sus columnas, pero coinciden en una columna clave, como un identificador de transacción. En otros casos, dos filas pueden parecer muy similares, aunque representen operaciones reales distintas.

Por ejemplo, dos clientes podrían comprar el mismo producto, en la misma sucursal, el mismo día y al mismo precio. Si esas ventas tienen identificadores diferentes, no deberíamos eliminarlas automáticamente solo porque se parecen.

Por eso, antes de eliminar duplicados necesitamos distinguir entre:

```text
filas exactamente repetidas
filas con identificadores repetidos
filas parecidas, pero posiblemente válidas
```

Esta diferencia es importante. Detectar duplicados es una tarea técnica, pero decidir qué hacer con ellos requiere interpretar qué representa cada fila dentro del dataset.

## Detectar duplicados completos

Para detectar filas duplicadas podemos usar el método `duplicated()`.

Cuando aplicamos `duplicated()` sobre un `DataFrame`, Pandas compara las filas completas. Si una fila ya apareció antes con los mismos valores en todas las columnas, la marca como duplicada.

Veamos qué ocurre con nuestro dataset.

In [62]:
df.duplicated()

,0
0,False
1,False
2,False
3,False
4,True
5,False
6,False
7,False
8,False
9,False


La salida es una serie booleana.

Cada valor indica si la fila correspondiente se considera duplicada o no. `False` significa que esa fila no fue detectada como duplicada. `True` significa que esa fila repite una combinación de valores que ya había aparecido antes.

Como una serie booleana completa puede ser difícil de leer, normalmente contamos cuántos duplicados fueron detectados.

In [63]:
df.duplicated().sum()

np.int64(2)

El resultado indica cuántas filas están duplicadas de manera completa.

Recordemos que, por defecto, Pandas conserva la primera aparición de una fila y marca como duplicadas las repeticiones posteriores. Por eso, si una fila aparece dos veces, solo la segunda aparición se marca como duplicada.

## Ver las filas duplicadas

Contar duplicados nos da una primera medida del problema, pero no alcanza para decidir qué hacer.

Antes de eliminar filas, conviene observar cuáles son los registros involucrados. Para eso podemos usar la máscara booleana que produce `duplicated()`.

Si escribimos:

```python
df[df.duplicated()]
```

Pandas muestra las filas que fueron marcadas como duplicadas.

Recordemos que, por defecto, `duplicated()` marca como duplicadas las repeticiones posteriores, pero no la primera aparición. Por eso, en esta vista veremos solamente la segunda aparición de cada fila repetida.

In [64]:
df[df.duplicated()]

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
4,T004,Te,2,900,Sur,2024-04-02
11,T010,Tostado,1,2500,Sur,2024-04-04


Esta salida muestra las filas que Pandas considera duplicadas completas.

Sin embargo, a veces queremos ver tanto la fila original como su repetición. Eso facilita la comparación y permite revisar el problema con más claridad.

Para eso podemos usar el parámetro `keep=False`.

In [65]:
df[df.duplicated(keep=False)]

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
3,T004,Te,2,900,Sur,2024-04-02
4,T004,Te,2,900,Sur,2024-04-02
10,T010,Tostado,1,2500,Sur,2024-04-04
11,T010,Tostado,1,2500,Sur,2024-04-04


Con `keep=False`, Pandas marca como duplicadas todas las apariciones de las filas repetidas, incluyendo la primera.

Esta vista es más útil para revisar duplicados, porque nos permite ver los grupos completos de filas repetidas.

En nuestro dataset aparecen dos grupos de duplicados completos. Cada grupo contiene una fila original y una repetición exacta.

Antes de eliminar duplicados, conviene observar estos casos y confirmar que realmente se trata de repeticiones accidentales.

## Eliminar duplicados completos

Después de detectar y revisar las filas duplicadas, podemos decidir eliminarlas.

Para eliminar duplicados completos en Pandas usamos `drop_duplicates()`.

Por defecto, este método conserva la primera aparición de cada fila repetida y elimina las repeticiones posteriores.

In [66]:
df_sin_duplicados = df.drop_duplicates()

df_sin_duplicados

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
0,T001,Cafe,2,1200,Centro,2024-04-01
1,T002,Medialuna,3,800,Centro,2024-04-01
2,T003,Cafe,1,1200,Norte,2024-04-01
3,T004,Te,2,900,Sur,2024-04-02
5,T005,Tostado,1,2500,Centro,2024-04-02
6,T006,Cafe,2,1200,Centro,2024-04-03
7,T007,Jugo,1,1500,Norte,2024-04-03
8,T008,Cafe,2,1200,Centro,2024-04-03
9,T009,Medialuna,3,800,Centro,2024-04-04
10,T010,Tostado,1,2500,Sur,2024-04-04


Ahora el nuevo `DataFrame` conserva una sola aparición de cada fila duplicada.

Es importante notar que guardamos el resultado en una nueva variable llamada `df_sin_duplicados`. De esta manera conservamos `df` como dataset original y podemos comparar antes y después.

Podemos revisar las dimensiones de ambos `DataFrames`.

In [67]:
print("Tamaño original:")
print(df.shape)

print()

print("Tamaño sin duplicados completos:")
print(df_sin_duplicados.shape)

Tamaño original:
(15, 6)

Tamaño sin duplicados completos:
(13, 6)


También podemos contar nuevamente los duplicados completos en la versión limpia.


In [68]:
df_sin_duplicados.duplicated().sum()

np.int64(0)

Si el resultado es `0`, significa que ya no quedan filas duplicadas completas.

Esta verificación es fundamental. Eliminar duplicados no debería terminar en la aplicación del método. Después de transformar el dataset, necesitamos comprobar que el problema que queríamos resolver efectivamente fue tratado.

En este caso, eliminamos duplicados completos: filas que eran iguales en todas sus columnas. Más adelante veremos que también podemos analizar duplicados considerando solo algunas columnas.

## Duplicados según una columna específica

Hasta ahora buscamos duplicados completos, es decir, filas idénticas en todas sus columnas.

Pero en muchos datasets hay columnas que deberían identificar de manera única cada registro. En nuestro ejemplo, esa columna es `id_transaccion`.

Si cada fila representa una transacción, entonces cada `id_transaccion` debería aparecer una sola vez. Si un mismo identificador aparece más de una vez, puede ser una señal de problema, aunque las filas no sean idénticas en todas sus columnas.

Podemos revisar duplicados considerando solamente la columna `id_transaccion`.

In [69]:
df.duplicated(subset=["id_transaccion"]).sum()

np.int64(2)

El parámetro `subset` permite indicar qué columnas queremos usar para detectar duplicados.

En este caso, Pandas no compara todas las columnas del `DataFrame`. Solo revisa si el valor de `id_transaccion` ya apareció antes.

Podemos ver las filas involucradas usando nuevamente `keep=False`.

In [70]:
df[df.duplicated(subset=["id_transaccion"], keep=False)]

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
3,T004,Te,2,900,Sur,2024-04-02
4,T004,Te,2,900,Sur,2024-04-02
10,T010,Tostado,1,2500,Sur,2024-04-04
11,T010,Tostado,1,2500,Sur,2024-04-04


Esta salida muestra todas las filas cuyos identificadores de transacción aparecen más de una vez.

En nuestro dataset, estos casos coinciden con duplicados completos. Pero en datasets reales podría ocurrir que dos filas compartan el mismo identificador y tengan diferencias en alguna otra columna. Eso requeriría una revisión más cuidadosa, porque podría tratarse de una carga duplicada con algún campo modificado o de un error en el identificador.

Por eso, revisar duplicados por columnas específicas es una parte importante del diagnóstico.

## Filas parecidas no siempre son duplicadas

Una parte importante del análisis de duplicados consiste en no eliminar registros solo porque se parecen.

En nuestro dataset hay algunas filas que comparten varios valores: mismo producto, misma cantidad, mismo precio unitario, misma sucursal y misma fecha. Sin embargo, si tienen distinto `id_transaccion`, podrían representar ventas reales distintas.

Veamos un ejemplo.

In [71]:
df[
    (df["producto"] == "Cafe")
    & (df["cantidad"] == 2)
    & (df["precio_unitario"] == 1200)
    & (df["sucursal"] == "Centro")
]

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
0,T001,Cafe,2,1200,Centro,2024-04-01
6,T006,Cafe,2,1200,Centro,2024-04-03
8,T008,Cafe,2,1200,Centro,2024-04-03
12,T011,Cafe,2,1200,Centro,2024-04-05
13,T012,Cafe,2,1200,Centro,2024-04-05


En esta salida pueden aparecer filas muy parecidas.

La pregunta importante es si esas filas representan una repetición accidental o ventas reales distintas.

Si tienen identificadores de transacción diferentes, no deberíamos eliminarlas automáticamente. Podría tratarse de dos clientes distintos que compraron el mismo producto, en la misma sucursal, con la misma cantidad y al mismo precio.

Este ejemplo muestra por qué la detección de duplicados no debe basarse solamente en una impresión visual.

En datasets de transacciones, el identificador suele ser clave. Si dos filas comparten todos los datos pero tienen distinto identificador, conviene revisar el contexto antes de decidir. Si dos filas comparten el mismo identificador, la sospecha de duplicado es más fuerte.

## Duplicados según varias columnas

Además de buscar duplicados completos o duplicados por identificador, también podemos revisar duplicados usando un conjunto de columnas.

Esto puede ser útil cuando no tenemos una columna identificadora confiable o cuando queremos investigar registros que se parecen demasiado.

Por ejemplo, podríamos revisar si hay filas repetidas considerando estas columnas:

```text
producto
cantidad
precio_unitario
sucursal
fecha
```

Esta revisión no usa `id_transaccion`. Por eso, no nos dice automáticamente qué filas debemos eliminar. Solo nos ayuda a encontrar registros que comparten las mismas características principales de venta.


In [72]:
columnas_para_revisar = [
    "producto",
    "cantidad",
    "precio_unitario",
    "sucursal",
    "fecha"
]

df[df.duplicated(subset=columnas_para_revisar, keep=False)]

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
3,T004,Te,2,900,Sur,2024-04-02
4,T004,Te,2,900,Sur,2024-04-02
6,T006,Cafe,2,1200,Centro,2024-04-03
8,T008,Cafe,2,1200,Centro,2024-04-03
10,T010,Tostado,1,2500,Sur,2024-04-04
11,T010,Tostado,1,2500,Sur,2024-04-04
12,T011,Cafe,2,1200,Centro,2024-04-05
13,T012,Cafe,2,1200,Centro,2024-04-05


La salida muestra filas que coinciden en las columnas seleccionadas.

Algunas de esas filas pueden ser duplicados reales. Otras pueden ser ventas diferentes que casualmente tienen los mismos valores en esas columnas.

Por eso, esta revisión debe interpretarse con cuidado. Cuando usamos `subset`, estamos cambiando el criterio de duplicación. Ya no estamos preguntando si toda la fila está repetida, sino si se repite una combinación específica de columnas.

Este tipo de análisis puede servir como señal de alerta, pero no reemplaza la interpretación del contexto.

Si el dataset tiene un identificador confiable, como `id_transaccion`, ese identificador suele ser una referencia muy importante. Si no lo tiene, debemos apoyarnos en varias columnas y revisar los casos con más atención.

## Qué significa el parámetro keep

Cuando usamos `duplicated()` o `drop_duplicates()`, Pandas necesita decidir qué aparición de una fila repetida debe conservarse y cuál debe marcarse como duplicada.

Para eso existe el parámetro `keep`.

Por defecto, `keep="first"`. Eso significa que Pandas conserva la primera aparición y considera duplicadas las apariciones posteriores.

Podemos verlo con `duplicated()`:

In [73]:
df.duplicated(keep="first")

,0
0,False
1,False
2,False
3,False
4,True
5,False
6,False
7,False
8,False
9,False


Si queremos marcar como duplicada la primera aparición y conservar la última, podemos usar `keep="last"`.

In [74]:
df.duplicated(keep="last")

,0
0,False
1,False
2,False
3,True
4,False
5,False
6,False
7,False
8,False
9,False


Y si queremos marcar todas las apariciones de las filas repetidas, usamos `keep=False`.

In [75]:
df.duplicated(keep=False)

,0
0,False
1,False
2,False
3,True
4,True
5,False
6,False
7,False
8,False
9,False


Esta última opción es muy útil para revisar duplicados, porque permite ver tanto la fila original como sus repeticiones.

También podemos usar `keep` con `drop_duplicates()`.

Por ejemplo, si eliminamos duplicados conservando la primera aparición:

In [76]:
df.drop_duplicates(keep="first")

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
0,T001,Cafe,2,1200,Centro,2024-04-01
1,T002,Medialuna,3,800,Centro,2024-04-01
2,T003,Cafe,1,1200,Norte,2024-04-01
3,T004,Te,2,900,Sur,2024-04-02
5,T005,Tostado,1,2500,Centro,2024-04-02
6,T006,Cafe,2,1200,Centro,2024-04-03
7,T007,Jugo,1,1500,Norte,2024-04-03
8,T008,Cafe,2,1200,Centro,2024-04-03
9,T009,Medialuna,3,800,Centro,2024-04-04
10,T010,Tostado,1,2500,Sur,2024-04-04


Si quisiéramos conservar la última aparición:

In [77]:
df.drop_duplicates(keep="last")

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
0,T001,Cafe,2,1200,Centro,2024-04-01
1,T002,Medialuna,3,800,Centro,2024-04-01
2,T003,Cafe,1,1200,Norte,2024-04-01
4,T004,Te,2,900,Sur,2024-04-02
5,T005,Tostado,1,2500,Centro,2024-04-02
6,T006,Cafe,2,1200,Centro,2024-04-03
7,T007,Jugo,1,1500,Norte,2024-04-03
8,T008,Cafe,2,1200,Centro,2024-04-03
9,T009,Medialuna,3,800,Centro,2024-04-04
11,T010,Tostado,1,2500,Sur,2024-04-04


En este dataset, como los duplicados completos son exactamente iguales, conservar la primera o la última aparición no cambia los valores finales. Pero en otros casos puede importar mucho, especialmente si las filas tienen el mismo identificador y alguna diferencia en otras columnas.

Por eso, antes de eliminar duplicados, conviene revisar qué criterio de conservación estamos usando.

## Verificar después de eliminar duplicados

Después de eliminar duplicados, siempre conviene verificar el resultado.

En limpieza de datos, no alcanza con aplicar una transformación. También necesitamos comprobar qué cambió y si el problema que queríamos resolver efectivamente quedó tratado.

Primero podemos crear una versión sin duplicados completos:

In [78]:
df_sin_duplicados = df.drop_duplicates()

df_sin_duplicados

,id_transaccion,producto,cantidad,precio_unitario,sucursal,fecha
0,T001,Cafe,2,1200,Centro,2024-04-01
1,T002,Medialuna,3,800,Centro,2024-04-01
2,T003,Cafe,1,1200,Norte,2024-04-01
3,T004,Te,2,900,Sur,2024-04-02
5,T005,Tostado,1,2500,Centro,2024-04-02
6,T006,Cafe,2,1200,Centro,2024-04-03
7,T007,Jugo,1,1500,Norte,2024-04-03
8,T008,Cafe,2,1200,Centro,2024-04-03
9,T009,Medialuna,3,800,Centro,2024-04-04
10,T010,Tostado,1,2500,Sur,2024-04-04


Ahora podemos comparar el tamaño del dataset original con el tamaño del dataset sin duplicados:

In [79]:
print("Tamaño original:")
print(df.shape)

print()

print("Tamaño sin duplicados:")
print(df_sin_duplicados.shape)

Tamaño original:
(15, 6)

Tamaño sin duplicados:
(13, 6)


También podemos calcular cuántas filas se eliminaron:

In [80]:
filas_eliminadas = df.shape[0] - df_sin_duplicados.shape[0]

filas_eliminadas

2

Y podemos verificar que ya no queden duplicados completos:

In [81]:
df_sin_duplicados.duplicated().sum()

np.int64(0)

Si el resultado es `0`, significa que no quedan filas duplicadas completas.

También podemos revisar si siguen existiendo identificadores de transacción repetidos:

In [82]:
df_sin_duplicados.duplicated(subset=["id_transaccion"]).sum()

np.int64(0)

En este caso, si eliminamos duplicados completos y los identificadores repetidos correspondían a filas exactamente iguales, el conteo también debería quedar en `0`.

Estas verificaciones son importantes porque nos permiten confirmar el efecto de la limpieza. Además, nos ayudan a documentar el proceso:

```text
cuántas filas había antes
cuántas filas quedaron después
cuántas filas fueron eliminadas
si todavía quedan duplicados según el criterio elegido
```

Eliminar duplicados no debe ser una acción invisible. Debemos poder explicar qué criterio usamos y qué impacto tuvo sobre el dataset.


## Errores frecuentes al trabajar con duplicados

Al trabajar con duplicados, uno de los errores más frecuentes es eliminar filas sin revisarlas antes.

Aunque `drop_duplicates()` es una herramienta muy útil, no deberíamos aplicarla automáticamente sin observar qué registros está eliminando. En algunos casos, las filas repetidas pueden ser errores claros. En otros, dos registros parecidos pueden representar eventos reales distintos.

Otro error frecuente es pensar que solo existen duplicados cuando toda la fila es exactamente igual. A veces el problema está en una columna específica, como un identificador de transacción repetido. Por eso, además de buscar duplicados completos, puede ser necesario revisar duplicados usando `subset`.

Por ejemplo:

In [83]:
df.duplicated(subset=["id_transaccion"]).sum()

np.int64(2)

Esta instrucción no compara toda la fila. Solo revisa si hay identificadores de transacción repetidos.

También debemos tener cuidado con el criterio usado para conservar registros. Por defecto, Pandas usa `keep="first"`, es decir, conserva la primera aparición y elimina o marca las siguientes. En algunos casos podría convenir conservar la última aparición, especialmente si el dataset fue actualizado y la fila más reciente contiene información corregida.

Otro error posible es confundir filas parecidas con duplicados. Dos ventas del mismo producto, con la misma cantidad, el mismo precio, la misma sucursal y la misma fecha pueden parecer repetidas, pero si tienen identificadores distintos podrían ser ventas reales diferentes.

Por eso, antes de eliminar duplicados conviene seguir esta rutina:

```text
detectar duplicados
observar las filas involucradas
definir el criterio de duplicación
eliminar solo si corresponde
verificar el resultado
```

El objetivo no es borrar registros hasta que no quede ninguna repetición aparente. El objetivo es decidir, con criterio, qué registros representan duplicaciones reales y cuáles deben conservarse.


## Resumen del capítulo

En este capítulo trabajamos con duplicados dentro de un `DataFrame`.

Comenzamos usando un dataset sintético de ventas, construido especialmente para mostrar distintos casos posibles: filas exactamente repetidas, identificadores de transacción repetidos y filas parecidas que no necesariamente debían eliminarse.

La idea inicial fue que un duplicado puede entenderse de distintas maneras. A veces dos filas son duplicadas porque todos sus valores son exactamente iguales. Otras veces, el problema aparece porque se repite una columna importante, como `id_transaccion`. También vimos que dos filas pueden ser muy parecidas y, aun así, representar ventas reales distintas.

Para detectar duplicados completos usamos:

```python
df.duplicated()
```

y para contarlos:

```python
df.duplicated().sum()
```

Luego observamos las filas duplicadas. Primero vimos solamente las repeticiones posteriores:

```python
df[df.duplicated()]
```

y después usamos `keep=False` para ver tanto la fila original como sus repeticiones:

```python
df[df.duplicated(keep=False)]
```

Esta revisión fue importante porque antes de eliminar registros conviene observar qué filas están involucradas.

Después usamos `drop_duplicates()` para eliminar duplicados completos:

```python
df_sin_duplicados = df.drop_duplicates()
```

También verificamos el resultado comparando el tamaño del dataset antes y después de la eliminación, contando cuántas filas se eliminaron y revisando si todavía quedaban duplicados.

Más adelante analizamos duplicados según una columna específica:

```python
df.duplicated(subset=["id_transaccion"]).sum()
```

Esto nos permitió mostrar que, en muchos datasets, no alcanza con buscar filas completamente idénticas. Si una columna funciona como identificador único, su repetición puede ser una señal de problema aunque el resto de la fila no sea exactamente igual.

También revisamos duplicados según varias columnas usando `subset`. Esta técnica puede ser útil cuando no tenemos un identificador confiable o cuando queremos investigar registros demasiado parecidos. Sin embargo, vimos que esa revisión debe interpretarse con cuidado: filas parecidas no siempre son duplicadas reales.

Finalmente trabajamos con el parámetro `keep`, que permite controlar qué aparición se conserva o se marca como duplicada:

```python
df.duplicated(keep="first")
df.duplicated(keep="last")
df.duplicated(keep=False)
```

La idea principal de este capítulo fue:

```text
Detectar duplicados es una tarea técnica, pero decidir si eliminarlos requiere interpretar qué representa cada fila.
```

Eliminar duplicados puede mejorar la calidad de un dataset, pero solo si tenemos claro qué criterio estamos usando y qué registros estamos quitando.


## Próximo paso

Ya trabajamos con valores faltantes y duplicados, dos problemas muy frecuentes en la limpieza de datos.

El siguiente paso será volver sobre la limpieza de textos y categorías, pero ahora dentro de esta etapa de preparación de datos. Vamos a revisar espacios innecesarios, diferencias entre mayúsculas y minúsculas, valores escritos de forma inconsistente y reemplazos necesarios para unificar categorías.

Ese trabajo será importante porque muchas columnas categóricas pueden parecer correctas a simple vista, pero generar conteos fragmentados si los valores no están escritos de manera uniforme.